# Week 07 - Regression Baselines

**Project:** AI/ML-Augmented Black-Scholes Framework
**Mandatory output of master plan section 11.7.**

The milestone:

> The student must understand why a linear model cannot generally reproduce
> the full nonlinear Black-Scholes structure.

This notebook does not try to make the linear model work. It predicts, from
convexity, **where** and **in which direction** the model must fail, then
checks whether it does.

### One thing to be clear about first

The target was generated by the Black-Scholes formula with no noise, and the
benchmark is that same formula. The benchmark error is exactly zero and **no
model can beat it**. A regression that appeared to would be a bug. The purpose
of the synthetic stage is a calibrated failure and a floor for Week 7, not a
contest.

In [ ]:
import sys, os

# Run from the repository root: the modules live there, and every path in the
# project (data/, results/, ml_plots/) is relative to it. Without the chdir a
# notebook launched from notebooks/ would quietly write a second copy of the
# dataset into notebooks/data/.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working directory:", os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 3.8), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

from ml_dataset import (SEED, load_dataset, get_xy,
                        FEATURES_BASIC, FEATURES_EXTENDED, TARGET, build_dataset)
from linear_models import (Standardiser, ALPHA_GRID, fit_ols, fit_ridge,
                           select_ridge_alpha, metrics)

K_REF = 100.0        # errors are quoted in currency units at this strike
df = load_dataset()
print(len(df), "rows")
print(df["split"].value_counts().to_string())

## 1. Scaling, fitted on training rows only

`z = (x - mu) / s`, with `mu` and `s` from the **training split alone**
(master plan 5.12, 5.13). Fitting the scaler on everything would leak test
information into training through the mean and standard deviation.

A free consistency check comes with it: OLS is invariant to any invertible
linear rescaling of the features, so its predictions must be identical with
and without the scaler. If they are not, the scaler has been applied in the
wrong direction somewhere.

In [ ]:
Xtr, ytr = get_xy(df, FEATURES_BASIC, split="train")
Xva, yva = get_xy(df, FEATURES_BASIC, split="val")
Xte, yte = get_xy(df, FEATURES_BASIC, split="test")

sc = Standardiser().fit(Xtr)          # TRAIN ONLY
print("mu", np.round(sc.mu, 5))
print("s ", np.round(sc.s, 5))

a = fit_ols(Xtr, ytr).predict(Xtr)
b = fit_ols(sc.transform(Xtr), ytr).predict(sc.transform(Xtr))
print("\nOLS invariance to standardisation: max difference %.2e" % np.abs(a-b).max())

## 2. OLS and Ridge

Both are solved in closed form from the normal equations, so there is no
convergence tolerance to argue about. `linear_models.py` cross-checks both
against scikit-learn when it is installed.

Ridge `alpha` is chosen on the **validation** split. The test split is not
touched until section 5.

In [ ]:
Ztr, Zva, Zte = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)

ols = fit_ols(Ztr, ytr, feature_names=FEATURES_BASIC)
alpha, ridge, tab = select_ridge_alpha(Ztr, ytr, Zva, yva,
                                       feature_names=FEATURES_BASIC, verbose=True)

print("\nOLS coefficients on standardised features:")
print("  intercept %+.6f" % ols.intercept)
for nm, c in zip(FEATURES_BASIC, ols.coef):
    print("  %-7s   %+.6f" % (nm, c))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot([t[0] for t in tab], [t[1]*K_REF for t in tab], marker="o", ms=3)
ax[0].axvline(alpha, ls=":", color="black")
ax[0].set_xscale("log"); ax[0].set_xlabel(r"$\alpha$")
ax[0].set_ylabel("validation RMSE"); ax[0].set_title("alpha chosen on validation")

paths = np.array([fit_ridge(Ztr, ytr, al).coef for al in ALPHA_GRID])
for j, nm in enumerate(FEATURES_BASIC):
    ax[1].plot(ALPHA_GRID, paths[:, j], label=nm)
ax[1].set_xscale("log"); ax[1].axhline(0, color="black", lw=0.8)
ax[1].set_xlabel(r"$\alpha$"); ax[1].set_ylabel("coefficient")
ax[1].set_title("coefficient paths"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

Ridge buys nothing here, and the reason is worth stating. With 14,000
training rows and four features there is no variance problem to regularise.
The error is **bias**, and a penalty on the coefficients cannot touch bias.
That is a useful negative result to hold before Week 7 reaches for more
complex models.

## 3. Where the model fails - predicted first

`y = C/K` is convex in `m`. A least-squares line through a convex function
**over-predicts in the interior and under-predicts at both ends**: for
`y = x^2` on `[-1, 1]` the fitted line is `y = 1/3`, so the residual
`1/3 - x^2` is positive in the middle and negative at the ends.

So the prediction, written before looking:

1. mean residual **positive** near the money, **negative** in both wings;
2. `|residual|` **largest at short maturity**, where the surface bends most;
3. `|residual|` **positively correlated with Gamma**, which is the curvature
   in `S` and which the model never saw.

In [ ]:
X, y = get_xy(df, FEATURES_BASIC, split=None)
res = (ols.predict(sc.transform(X)) - y) * K_REF
d = df.copy(); d["residual"] = res

mb = [-0.61, -0.4, -0.2, -0.05, 0.05, 0.2, 0.4, 0.61]
g = d.groupby(pd.cut(d["m"], mb), observed=True)["residual"].agg(
        ["count", "mean", "std", lambda s: s.abs().mean()])
g.columns = ["n", "mean", "sd", "mean_abs"]
g.round(4)

In [ ]:
tb = [0.0, 0.08, 0.25, 0.5, 1.0, 1.5, 2.01]
gt = d.groupby(pd.cut(d["T"], tb), observed=True)["residual"].agg(
        ["count", "mean", "std", lambda s: s.abs().mean()])
gt.columns = ["n", "mean", "sd", "mean_abs"]
gt.round(4)

In [ ]:
gam = d["gamma"].to_numpy() * d["K"].to_numpy()
print("corr(|residual|, K*Gamma) = %+.4f   <- curvature in S" % np.corrcoef(np.abs(res), gam)[0,1])
print("corr(|residual|, T)       = %+.4f   <- shorter maturity, larger error"
      % np.corrcoef(np.abs(res), d["T"])[0,1])

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
s = d.sample(6000, random_state=SEED)
ax[0].axhline(0, color="black", lw=1)
ax[0].scatter(s["m"], s["residual"], s=3, alpha=0.3, color="#b5483d")
ax[0].set_xlabel("m"); ax[0].set_ylabel("residual (currency)")
ax[0].set_title("the convexity signature")

mbin = np.linspace(-0.6, 0.6, 25)
tbin = np.logspace(np.log10(d["T"].min()), np.log10(d["T"].max()), 20)
H, _, _ = np.histogram2d(d["m"], d["T"], bins=[mbin, tbin], weights=d["residual"])
C, _, _ = np.histogram2d(d["m"], d["T"], bins=[mbin, tbin])
with np.errstate(invalid="ignore"):
    Mn = np.where(C > 0, H / np.maximum(C, 1), np.nan)
lim = np.nanmax(np.abs(Mn))
im = ax[1].pcolormesh(mbin, tbin, Mn.T, cmap="RdBu_r", vmin=-lim, vmax=lim)
ax[1].set_yscale("log"); ax[1].set_xlabel("m"); ax[1].set_ylabel("T")
ax[1].set_title("mean residual over (m, T)")
plt.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.show()

The inverted parabola on the left is the convexity signature. It is
not noise around zero: it is a smooth, systematic, sign-changing bias, which
is exactly what a linear approximation to a convex function produces.

## 4. Is it the features or the functional form?

A large error has two possible explanations. Either `[m, T, r, sigma]` does
not contain enough information, or it does and a *linear map* cannot extract
it.

These are distinguishable. Expand the **same four features** to degree two -
adding no new information, only products of what is already there - and refit.
If the features were inadequate, this would not help.

In [ ]:
def expand(Z):
    cols, p = [Z], Z.shape[1]
    for i in range(p):
        for j in range(i, p):
            cols.append((Z[:, i] * Z[:, j])[:, None])
    return np.hstack(cols)

sc_q = Standardiser().fit(expand(Xtr))
ols_q = fit_ols(sc_q.transform(expand(Xtr)), ytr)

lin = metrics(yte, ols.predict(Zte), scale=K_REF)
quad = metrics(yte, ols_q.predict(sc_q.transform(expand(Xte))), scale=K_REF)
print("test RMSE  linear    %.4f" % lin["RMSE"])
print("test RMSE  quadratic %.4f" % quad["RMSE"])
print("improvement factor   %.1fx  with no new information added"
      % (lin["RMSE"] / quad["RMSE"]))

## 5. Final evaluation - the test split, touched once

Everything above used training and validation only. Both models are now
frozen, so the test split can be scored (master plan 5.10.3).

In [ ]:
rows = []
for label, pred in [
        ("OLS  [m,T,r,sigma]",      ols.predict(Zte)),
        ("Ridge [m,T,r,sigma]",     ridge.predict(Zte)),
        ("OLS  quadratic (same 4)", ols_q.predict(sc_q.transform(expand(Xte)))),
]:
    m = metrics(yte, pred, scale=K_REF); m["model"] = label; rows.append(m)

Xe, ye = get_xy(df, FEATURES_EXTENDED, split="train")
sce = Standardiser().fit(Xe)
olse = fit_ols(sce.transform(Xe), ye)
Xet, yet = get_xy(df, FEATURES_EXTENDED, split="test")
m = metrics(yet, olse.predict(sce.transform(Xet)), scale=K_REF)
m["model"] = "OLS  extended (+Greeks)"; rows.append(m)

m = metrics(yte, yte, scale=K_REF); m["model"] = "Black-Scholes (benchmark)"
rows.append(m)

pd.DataFrame(rows)[["model", "MAE", "RMSE", "R2", "max_abs_error"]].round(4)

## 6. What this establishes

- The linear baseline reaches an `R^2` around 0.82, which looks respectable
  and is precisely why `R^2` is the wrong statistic here. The residual
  structure, not the summary number, is the finding.
- The failure is **the functional form, not the features**: a degree-two
  expansion of the same four variables cuts test RMSE by roughly a factor of
  five without adding any information.
- Ridge does nothing, because the error is bias rather than variance.
- These numbers are the floor for **Stage Gate 2 on 17 November**. A tree
  ensemble or a neural network must beat them by enough to justify its
  complexity - and the honest bar is the quadratic model, not the linear one,
  since polynomial regression is nearly free.